In [ ]:
!pip install transformers datasets sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.0 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
import faiss
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------
# MODELS
# ----------------------------
GEN_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

MAX_NEW_TOKENS = 600

# ----------------------------
# LOAD DATASET
# ----------------------------
ds = load_dataset("Eloquent/Voight-Kampff", "test")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

task-vk-test-2025.json: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/2 [00:00<?, ? examples/s]

In [ ]:
# ----------------------------
# LOAD GENERATOR
# ----------------------------
print("Loading Generator...")
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

Loading Generator...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
# ----------------------------
# SIMPLE HUMAN CORPUS (Wikipedia)
# ----------------------------
wiki = load_dataset("cc_news", split="train[:500]")

documents = []
for row in wiki:
    text = row["text"]
    if len(text.split()) > 200:
        documents.append(text[:1500])  # truncate

print("Corpus size:", len(documents))

# ----------------------------
# BUILD VECTOR INDEX
# ----------------------------
embedder = SentenceTransformer(EMBED_MODEL)

doc_embeddings = embedder.encode(documents, convert_to_numpy=True)

dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

# ----------------------------
# RETRIEVAL FUNCTION
# ----------------------------
def retrieve_context(query, k=3):
    query_emb = embedder.encode([query])
    distances, indices = index.search(query_emb, k)
    return [documents[i] for i in indices[0]]

# ----------------------------
# PROMPT BUILDERS
# ----------------------------
def build_baseline_prompt(base_prompt, genre, content):
    return f"""
{base_prompt}

Genre: {genre}

Content:
{content}

Write ~500 words.
"""

def build_rag_prompt(base_prompt, genre, content):
    query = base_prompt + " " + genre
    retrieved = retrieve_context(query, k=3)
    context_block = "\n\n".join(retrieved)

    return f"""
Below are examples of human writing style:

{context_block}

---

Now write a new original text.

{base_prompt}

Genre: {genre}

Content:
{content}

Write ~500 words.
Do not copy examples.
"""

# ----------------------------
# GENERATION
# ----------------------------
def generate(prompt):
    output = generator(
        prompt,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=1.1,
        top_p=0.95
    )
    return output[0]["generated_text"][len(prompt):]

# ----------------------------
# DETECTOR (Curvature Approximation)
# ----------------------------
def get_logprob(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    return -outputs.loss.item()

def curvature_score(text):
    original_lp = get_logprob(text)

    perturbed = text.replace(".", ". ")  # simple minimal perturbation
    perturbed_lp = get_logprob(perturbed)

    return original_lp - perturbed_lp

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00005.parquet:   0%|          | 0.00/211M [00:00<?, ?B/s]

plain_text/train-00001-of-00005.parquet:   0%|          | 0.00/234M [00:00<?, ?B/s]

plain_text/train-00002-of-00005.parquet:   0%|          | 0.00/219M [00:00<?, ?B/s]

plain_text/train-00003-of-00005.parquet:   0%|          | 0.00/245M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# ----------------------------
# RUN EXPERIMENT
# ----------------------------
task_data = ds["test"][0]["voight-kampfftesttopics"]

base_prompt = task_data["prompt"]
topic = task_data["topics"][0]

genre = topic["Genre and Style"]
content = topic["Content"]

print("\nGenerating BASELINE...")
baseline_prompt = build_baseline_prompt(base_prompt, genre, content)
baseline_text = generate(baseline_prompt)
baseline_score = curvature_score(baseline_text)

print("Baseline Score:", baseline_score)

print("\nGenerating RAG...")
rag_prompt = build_rag_prompt(base_prompt, genre, content)
rag_text = generate(rag_prompt)
rag_score = curvature_score(rag_text)

print("RAG Score:", rag_score)

print("\nComparison:")
print("Baseline:", baseline_score)
print("RAG:", rag_score)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_p', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generating BASELINE...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Baseline Score: 0.02698993682861328

Generating RAG...
RAG Score: 0.004931807518005371

Comparison:
Baseline: 0.02698993682861328
RAG: 0.004931807518005371
